# ⚡ GridSentinel — run it on free Google Colab

**GridSentinel** learns the normal rhythm of a power grid and catches attackers who
feed it *fake sensor readings* (false data injection), including the stealthy kind
that classic alarms miss.

Everything here is **simulated** — no real substation or network is touched.

This notebook runs the whole thing on Colab's free CPU in ~1–2 minutes.
Just press **Runtime → Run all**.


## 1. Get the code

Replace the URL with *your* GitHub repo if you forked it.

In [ ]:
# clone the project (edit the URL to your fork if needed)
import os
REPO = "https://github.com/Lohit-18/GridSentinel.git"
if not os.path.exists("GridSentinel"):
    !git clone -q $REPO || echo "clone failed - you can also upload the project folder"
%cd GridSentinel
!pip -q install -e . 2>/dev/null || pip -q install numpy scipy scikit-learn pandas matplotlib rich
import sys; sys.path.insert(0, "src")
print("ready")

## 2. Train the detector and score an unseen attack campaign

In [ ]:
from gridsentinel.eval.experiments import run_experiment
result = run_experiment("ieee14", train_steps=2000, test_steps=1600, verbose=True)

import pandas as pd
rows = []
for d in result.to_json()["detector_order"]:
    m = result.detector_metrics.get(d)
    if m: rows.append({"detector": result.detector_labels[d],
                       "precision": round(m["precision"],3), "recall": round(m["recall"],3),
                       "F1": round(m["f1"],3), "FPR": round(m["false_positive_rate"],3),
                       "ROC_AUC": round(m.get("roc_auc", float("nan")),3)})
pd.DataFrame(rows)

## 3. The money shot — a stealth attack that physics can't see

Top: the classic χ² statistic never crosses its alarm line.
Bottom: GridSentinel's ensemble fires on the same window.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from gridsentinel.viz.plots import generate_all_figures
figs = generate_all_figures(result, "results/figures")
from IPython.display import Image, display
display(Image("results/figures/timeseries_detection.png"))

## 4. The rest of the evidence

In [ ]:
for f in ["detector_comparison","per_attack_recall","roc_pr","feature_importance","confusion_matrices"]:
    display(Image(f"results/figures/{f}.png"))

## 5. Build the shareable HTML report (optional)

In [ ]:
from gridsentinel.report.html_report import build_report
path = build_report(result, figs, [], "results/report.html")
print("wrote", path)
# download it:
try:
    from google.colab import files; files.download(path)
except Exception:
    print("open results/report.html from the file browser on the left")

## 6. Scale it up (optional, ~1 min)

Same detector across bigger grids.

In [ ]:
from gridsentinel.eval.experiments import run_scaling_study
pd.DataFrame(run_scaling_study(steps=1000))

---
*GridSentinel v1.0 · Apache-2.0 · a research/education project, not a certified relay.*